# Module 3: Sequential Chain — Production Deployment

Deploy the three specialist runtimes to Amazon Bedrock AgentCore Runtime (A2A protocol).  
The sequential coordination runs locally via `chain.py` using Strands `GraphBuilder`.

---

## Step 1: Install dependencies

In [ ]:
%pip install "strands-agents[a2a]>=1.52.0" bedrock-agentcore boto3 httpx

# ⚠️  If this cell installs new packages (e.g. a2a-sdk), restart the kernel
# before continuing: Kernel → Restart Kernel, then re-run from Step 2 onward.
# You do NOT need to re-deploy — the ARNs in .env_arns are still valid.

---

## Step 2: Set up execution roles

Creates the IAM execution roles for the AgentCore runtimes (idempotent — safe to re-run).

In [ ]:
import sys, os, boto3
sys.path.insert(0, "../../shared")
import deploy_utils as u

session = u.get_session()
account = u.get_account(session)
bucket  = u.code_bucket_name(account, u.REGION)
iam     = session.client("iam", region_name=u.REGION)

# Create runtime role (Bedrock, Logs, S3) — shared by all three specialists
runtime_role_arn = u.ensure_runtime_role(
    iam, f"workshop-agentcore-m3-runtime-role",
    account, u.REGION, bucket,
)
os.environ["AGENTCORE_RUNTIME_ROLE_ARN"] = runtime_role_arn

---

## Step 3: Deploy

Deploys the three specialist runtimes (~3-5 min). The coordination layer (`chain.py`) runs locally — no orchestrator runtime is deployed.

In [ ]:
!python deploy.py --name-prefix m3

In [ ]:
import os

# Read the three specialist ARNs saved by deploy.py and set them as env vars
with open(".env_arns", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith("export "):
            key, _, val = line[len("export "):].partition("=")
            os.environ[key.strip()] = val.strip()

RESEARCHER_ARN  = os.environ["RESEARCHER_RUNTIME_ARN"]
ANALYST_ARN     = os.environ["ANALYST_RUNTIME_ARN"]
SYNTHESIZER_ARN = os.environ["SYNTHESIZER_RUNTIME_ARN"]

---

## Step 4: Run the chain

The ARNs are now set as env vars. `chain.py` uses Strands `GraphBuilder` to call the three specialist runtimes in order: Researcher → Analyst → Synthesizer.

In [ ]:
import sys
sys.path.insert(0, ".")   # chain.py and a2a_utils.py are in the same folder

from chain import run_chain

BRIEF = (
    "NovaCart Premium Tier: Options A ($19.99/mo invite-only), "
    "B ($14.99/mo 5% pilot), C ($12.99/mo full launch). "
    "Target: +15% CLV in 6 months. Budget: $2M."
)

---

## Step 5: Observability

After running `chain.py`, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- One span per specialist runtime call (Researcher, Analyst, Synthesizer)
- Tool call spans nested under the Researcher span
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore instruments the specialists automatically.

---

## Step 6: Cleanup

Uncomment and run the cell below to delete all AWS resources created by this module.

In [ ]:
# Uncomment and run to delete all resources created by this module.

# !python cleanup.py --name-prefix m3

# Verify:
# import boto3, os
# REGION = os.environ.get("AWS_REGION", "us-east-1")
# remaining = boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes()
# print([rt["agentRuntimeName"] for rt in remaining.get("agentRuntimes", [])])